# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnzilaAhsan/week1-asm1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd, numpy as np, os

local_path = "../../data/raw/content_refresh_anonymized.csv"
if os.path.exists(local_path):
    df = pd.read_csv(local_path)
else:
    if not os.path.exists("internship"):
        get_ipython().system('git clone https://github.com/UnzilaAhsan/internship.git')
    df = pd.read_csv("internship/data/raw/content_refresh_anonymized.csv")

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(f"{len(df):,} rows | base rate (declining): {df['is_declining_label'].mean():.3f}")

for col in ["impressions_90d", "days_since_last_update", "ctr", "avg_position", "word_count"]:
    d = df[col].describe(percentiles=[.5, .9, .99])
    print(f"\n{col}: median={d['50%']:.2f}  p90={d['90%']:.2f}  p99={d['99%']:.2f}  max={d['max']:.2f}")

print(f"\nword_count missing: {df['word_count'].isna().sum():,} of {len(df):,} rows "
      f"({df['word_count'].isna().mean()*100:.1f}%)")


30,000 rows | base rate (declining): 0.542

impressions_90d: median=731.00  p90=12136.40  p99=73505.83  max=517715.00

days_since_last_update: median=20.00  p90=104.00  p99=106.00  max=373.00

ctr: median=0.07  p90=0.65  p99=8.33  max=100.00

avg_position: median=10.80  p90=36.80  p99=69.90  max=245.00

word_count: median=2877.00  p90=5327.00  p99=7292.00  max=9546.00

word_count missing: 7,699 of 30,000 rows (25.7%)


**Notes on the tails, in plain words:**
- `impressions_90d` is heavily right skewed: median 731 but the 99th percentile is 73,506 and the max is 517,715 - a handful of pages carry enormous volume. Any score built on raw impressions needs a log transform, or a few outlier pages will dominate every ranking.
- `ctr` is similarly skewed: median 0.07% but max 100% - some rows are likely single-digit impression counts where one click swings the ratio wildly. Bucketing (as I do below) is safer than treating raw CTR as a smooth number.
- `avg_position` runs 0-245, where **0 means no ranking data**, no "rank zero" - has to be filtered out before any position-based bucket, or it corrupts every "top N" cut.
- `word_count` is missing on **25.7%** of rows - not zero-filled, genuinely absent. Any thin-content signal has to explicitly handle the missingness rather than silently treating it as 0 words.       

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
# --- Test 1: staleness -> refresh flag ---
bins, labels = [-1, 30, 90, 180, 365, 100000], ["0-30d", "31-90d", "91-180d", "181-365d", "365d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels)
t1 = df.groupby("staleness_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
print("TEST 1 -- staleness vs decline:")
print(t1)
print("VERDICT: MIXED (two big buckets move the right way, small buckets don't cooperate)\n")

# --- Test 2: CTR-vs-position -> CTR-fix flag ---
good_pos = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 100)].copy()
ctr_bins, ctr_labels = [-1, 0.5, 1, 2, 5, 1000], ["<0.5%", "0.5-1%", "1-2%", "2-5%", "5%+"]
good_pos["ctr_bucket"] = pd.cut(good_pos["ctr"], bins=ctr_bins, labels=ctr_labels)
t2 = good_pos.groupby("ctr_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
print(f"TEST 2 -- CTR vs decline, among top-20 visible pages (n={len(good_pos):,}):")
print(t2)
print("VERDICT: CONFIRMED (falls almost monotonically as CTR rises)\n")

# --- Test 3: word count (thin content) -> thin_visible_page flag ---
wc_bins, wc_labels = [-1, 500, 1200, 2500, 100000], ["<500", "500-1200", "1200-2500", "2500+"]
df["wc_bucket"] = pd.cut(df["word_count"], bins=wc_bins, labels=wc_labels)
visible = df[df["impressions_90d"] >= 250]
t3 = visible.groupby("wc_bucket", observed=True)["is_declining_label"].agg(["mean", "count"])
print(f"TEST 3 -- word count vs decline, among visible pages (n={len(visible):,}):")
print(t3)
print("VERDICT: MIXED -- decline actually peaks in the 1200-2500 band (0.72), not the shortest "
      "band; the <500 band has essentially no rows (n=3) to judge at all. 'Thinner = more decline' "
      "is not supported here.")


TEST 1 -- staleness vs decline:
                      mean  count
staleness_bucket                 
0-30d             0.511377  20480
31-90d            0.588571    175
91-180d           0.611057   9171
181-365d          0.467456    169
365d+             0.600000      5
VERDICT: MIXED (two big buckets move the right way, small buckets don't cooperate)

TEST 2 -- CTR vs decline, among top-20 visible pages (n=15,091):
                mean  count
ctr_bucket                 
<0.5%       0.646890  12203
0.5-1%      0.510608   2121
1-2%        0.493130    655
2-5%        0.402062     97
5%+         0.533333     15
VERDICT: CONFIRMED (falls almost monotonically as CTR rises)

TEST 3 -- word count vs decline, among visible pages (n=19,386):
               mean  count
wc_bucket                 
500-1200   0.530120     83
1200-2500  0.716503   2642
2500+      0.629873  10953
VERDICT: MIXED -- decline actually peaks in the 1200-2500 band (0.72), not the shortest band; the <500 band has essentially

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Since it's the only confirmed signal above the threshold, we should focus on CTR-vs-position (low_ctr_visible_page) and make sure it works before relying on it. For that purpose, we need to carry out two checks: first, verify that it holds within each position sub-band (and not just when the data for positions 1 to 20 is pooled together), and second, check that it holds when looking at individual large clients (and not merely by taking the average, which could be influenced by one large client).

In [ ]:
# Does low-CTR-implies-more-decline hold within EACH position band separately?
print("Within each position band:")
for lo, hi, name in [(1, 5, "1-5"), (6, 10, "6-10"), (11, 20, "11-20")]:
    sub = good_pos[(good_pos["avg_position"] >= lo) & (good_pos["avg_position"] <= hi)]
    low = sub[sub["ctr"] < 0.5]["is_declining_label"]
    high = sub[sub["ctr"] >= 0.5]["is_declining_label"]
    print(f"  position {name}: n={len(sub):,}  low-CTR decline={low.mean():.3f} (n={len(low):,})  "
          f"higher-CTR decline={high.mean():.3f} (n={len(high):,})")

# Does it hold within each of the 5 largest clients individually?
print("\nWithin each of the 5 largest clients:")
for c in good_pos["client_id"].value_counts().head(5).index:
    sub = good_pos[good_pos["client_id"] == c]
    low = sub[sub["ctr"] < 0.5]["is_declining_label"].mean()
    high = sub[sub["ctr"] >= 0.5]["is_declining_label"].mean()
    print(f"  {c}: n={len(sub):,}  low-CTR decline={low:.3f}  higher-CTR decline={high:.3f}")

print("\nThe gap (low-CTR decline > higher-CTR decline) holds in all 3 position bands and all "
      "5 largest clients -- this isn't one client or one position range driving the pooled result.")


Within each position band:
  position 1-5: n=2,498  low-CTR decline=0.731 (n=1,757)  higher-CTR decline=0.462 (n=741)
  position 6-10: n=5,373  low-CTR decline=0.625 (n=4,360)  higher-CTR decline=0.508 (n=1,013)
  position 11-20: n=5,089  low-CTR decline=0.642 (n=4,283)  higher-CTR decline=0.543 (n=806)

Within each of the 5 largest clients:
  client_19581e27de: n=5,032  low-CTR decline=0.574  higher-CTR decline=0.406
  client_6208ef0f77: n=1,797  low-CTR decline=0.713  higher-CTR decline=0.460
  client_4e07408562: n=1,626  low-CTR decline=0.520  higher-CTR decline=0.319
  client_3fdba35f04: n=1,356  low-CTR decline=0.839  higher-CTR decline=0.721
  client_f369cb89fc: n=837  low-CTR decline=0.716  higher-CTR decline=0.631

The gap (low-CTR decline > higher-CTR decline) holds in all 3 position bands and all 5 largest clients -- this isn't one client or one position range driving the pooled result.


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can regard a low CTR on a page that is already visible and well ranked as a genuine and strong indicator for prioritising a review since this holds true across different position bands and in every case among the large clients examined, not just on average. Staleness and word count alone are not reliable enough to form the basis of any decision; although it is worth keeping an eye on them, they should not be the ones to determine a priority score since neither of them showed a clear pattern in this instance.

## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.